## Tello 드론 Mission Pad — go / curve / jump 파이썬 코드

---

## 📋 사전 준비
```bash
프로젝트 환경 만들기
  mkdir missionpad         : 프로젝트폴더생성
  cd missionpad            : 프로젝트폴더로 이동
  py --list                : 파이썬 버전 목록 확인
  py -3.14 -m venv venv    : 파이썬 가상환경 만들기
  venv\Scripts\activate    : 가상환경 시작( 종료는 : deactivate )
  pip install jupyterlab   : 주피터랩 패키지 설치
  pip install djitellopy   : DJI드론 API 설치
  jupyter lab --port=10888 : 주피터랩 시작
```

```
Mission Pad ID 규칙:
  mid 1~8 = m1~m8 (Tello SDK 기준)
  go/curve : 현재 위치 → 미션패드 기준 좌표로 이동
  jump     : 미션패드1 → 미션패드2 간 이동
```

---  


## 📊 명령어 파라미터 요약
<span style="position:left;display:inline-block;">
<table style="font-size:17px;">
    <tr align=center>
        <td>명령</td>
        <td>파라미터</td>
        <td>제한값</td>
    </tr>
    <tr>
        <td align=center>go</td>
        <td>x y z speed mid<br>미션패드 기준 좌표로 직선 이동</td>
        <td>xyz: -500~500<br>speed: 10~100</td>
    </tr>
    <tr>
        <td align=center>curve</td>
        <td>x1 y1 z1  x2 y2 z2  speed mid<br>경유점(x1y1z1) → 도착점(x2y2z2) 곡선 비행</td>
        <td>xyz: -500~500<br>speed: 10~60</td>
    </tr>
    <tr>
        <td align=center>jump</td>
        <td>x y z speed yaw mid1 mid2<br>미션패드1 → 미션패드2 간 이동</td>
        <td>xyz: -500~500<br>yaw: 0~360</td>
    </tr>
    <tr colspan=3><td>○ mid (Mission Pad ID) : 1~8 (m1~m8)<br>
○ 단위 : 좌표(cm), 속도(cm/s), 각도(도)</td></tr>
</table>
</span>

---


## 🟢 초급 (Beginner) — 소켓 직접 통신

### 1. 드론 제어할 UDP 소켓 생성

In [ ]:
# 드론제어 UDP 소켓 생성
import socket
import time

# ── 기본 설정 ──────────────────────────────────────
TELLO_IP   = '192.168.10.1'
TELLO_PORT = 8889
LOCAL_PORT = 9000

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind(('', LOCAL_PORT))
sock.settimeout(5)

def send(cmd):
    """명령 전송 및 응답 수신"""
    print(f"[전송] {cmd}")
    sock.sendto(cmd.encode('utf-8'), (TELLO_IP, TELLO_PORT))
    try:
        response, _ = sock.recvfrom(1024)
        res = response.decode('utf-8')
        print(f"[응답] {res}")
        return res
    except socket.timeout:
        print("[오류] 응답 없음")
        return None


### 2. 드론 SDK모드 활성화 및 이륙

In [ ]:
# SDK모드 활성화 및 이륙
# ── 초기화 ─────────────────────────────────────────
send('command')               # SDK 모드 진입
send('mon')                   # 미션패드 감지 활성화
time.sleep(1)
send('takeoff')               # 이륙
time.sleep(3)

### 3. go 명령

In [ ]:
# ── go 명령 ────────────────────────────────────────
# go x y z speed mid
# 미션패드(m1) 기준으로 x=0, y=0, z=50 위치로 이동
send('go 0 0 50 30 m3')
send('go 0 0 180 30 m3')
send('go 0 0 50 30 m3')
time.sleep(4)

### 4. curve 명령

In [ ]:
# ── curve 명령 ─────────────────────────────────────
# curve x1 y1 z1 x2 y2 z2 speed mid
# 경유점(50,0,50)을 거쳐 도착점(50,50,50)으로 곡선 비행
send('curve 0 50 100 50 0 180 30 m3')
time.sleep(5)

### 5. jump 명령

In [ ]:
# ── jump 명령 ──────────────────────────────────────
# jump x y z speed yaw mid1 mid2
# 미션패드1 → 미션패드2 로 이동 후 yaw 0도 회전
send('jump 50 0 50 30 0 m1 m3')
time.sleep(5)

In [ ]:
# ── 착륙 + 미션패드 비활성화 + 종료 ──
send('land')
send('moff')                  # 미션패드 감지 비활성화
sock.close()